# Cost Analysis
Calculates the real per-call cost of the search pipeline: embedding + reranking.
Uses tiktoken to count actual tokens from real DB samples, then projects monthly costs at different usage volumes.

In [8]:
%pip install tiktoken psycopg2-binary python-dotenv openai -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os, json
import tiktoken
import psycopg2
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

DATABASE_URL = os.getenv('DATABASE_URL').split('?')[0]
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def get_conn():
    return psycopg2.connect(DATABASE_URL, sslmode='require')

# Pricing (USD per 1M tokens)
EMBEDDING_PRICE_PER_1M   = 0.02   # text-embedding-3-small
NANO_INPUT_PRICE_PER_1M  = 0.20   # gpt-5.4-nano input
NANO_OUTPUT_PRICE_PER_1M = 1.25   # gpt-5.4-nano output

embed_enc = tiktoken.encoding_for_model('text-embedding-3-small')
nano_enc  = tiktoken.encoding_for_model('gpt-4o')  # same tokenizer as gpt-5.4-nano

def count_tokens(enc, text):
    return len(enc.encode(text, disallowed_special=()))

print('Setup complete')

Setup complete


In [10]:
# Sample 50 real issues to measure actual token counts
conn = get_conn()
cur = conn.cursor()
cur.execute("""
    SELECT i.title, i.body, l.name
    FROM issues i
    JOIN libraries l ON i.library_id = l.id
    WHERE i.body IS NOT NULL AND i.body != ''
    ORDER BY RANDOM()
    LIMIT 50
""")
samples = cur.fetchall()
cur.close()
conn.close()

print(f'Sampled {len(samples)} issues')

Sampled 50 issues


In [11]:
# Measure: embedding cost per search_bugs call
# A typical agent query is short — simulate with realistic query lengths
sample_queries = [
    'ImportError cannot import name BaseSettings from pydantic',
    'app.dependency_overrides not working in pytest',
    'celery task result backend not configured redis',
    'TypeError forward() got unexpected keyword argument head_mask',
    'SSLError CERTIFICATE_VERIFY_FAILED requests https',
]

query_tokens = [count_tokens(embed_enc, q) for q in sample_queries]
avg_query_tokens = sum(query_tokens) / len(query_tokens)

embed_cost_per_call = (avg_query_tokens / 1_000_000) * EMBEDDING_PRICE_PER_1M

print(f'Avg query tokens:        {avg_query_tokens:.1f}')
print(f'Embedding cost/call:     ${embed_cost_per_call:.8f}')
print(f'Embedding cost/1000 calls: ${embed_cost_per_call * 1000:.4f}')

Avg query tokens:        9.0
Embedding cost/call:     $0.00000018
Embedding cost/1000 calls: $0.0002


In [12]:
# Measure: reranking cost per search_bugs call
# Reranker receives: system prompt + query + 10 candidates (title + body[:300])

SYSTEM_PROMPT = "You are a relevance judge for a bug solutions search engine. Given a developer query and a list of GitHub issues, return the 1-based indices of the 3 most relevant issues in order of relevance."

rerank_input_tokens_list = []

for i in range(0, min(len(samples), 45), 10):
    # Build a realistic rerank prompt using 10 sampled issues as candidates
    batch = samples[i:i+10]
    query = sample_queries[i // 10 % len(sample_queries)]
    numbered = "\n".join([
        f"{j+1}. [{row[2]}] Title: {row[0]}\n   Body: {(row[1] or '')[:300]}"
        for j, row in enumerate(batch)
    ])
    full_prompt = f"{SYSTEM_PROMPT}\n\nQuery: {query}\n\nCandidates:\n{numbered}"
    rerank_input_tokens_list.append(count_tokens(nano_enc, full_prompt))

avg_rerank_input_tokens = sum(rerank_input_tokens_list) / len(rerank_input_tokens_list)
avg_rerank_output_tokens = 15  # just "2, 5, 8" wrapped in JSON schema

rerank_cost_per_call = (
    (avg_rerank_input_tokens / 1_000_000) * NANO_INPUT_PRICE_PER_1M +
    (avg_rerank_output_tokens / 1_000_000) * NANO_OUTPUT_PRICE_PER_1M
)

print(f'Avg rerank input tokens:   {avg_rerank_input_tokens:.1f}')
print(f'Avg rerank output tokens:  {avg_rerank_output_tokens}')
print(f'Rerank cost/call:          ${rerank_cost_per_call:.8f}')
print(f'Rerank cost/1000 calls:    ${rerank_cost_per_call * 1000:.4f}')

Avg rerank input tokens:   998.2
Avg rerank output tokens:  15
Rerank cost/call:          $0.00021839
Rerank cost/1000 calls:    $0.2184


In [13]:
# Total cost per search_bugs call (with and without reranking)
cost_without_rerank = embed_cost_per_call
cost_with_rerank    = embed_cost_per_call + rerank_cost_per_call

print(f'Cost per call (no rerank):   ${cost_without_rerank:.6f}')
print(f'Cost per call (with rerank): ${cost_with_rerank:.6f}')
print()

# Monthly projections
print(f'{"Volume":<20} {"No Rerank":>12} {"With Rerank":>12}')
print('-' * 46)
for volume in [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]:
    cost_a = volume * cost_without_rerank
    cost_b = volume * cost_with_rerank
    print(f'{volume:<20,} ${cost_a:>11.4f} ${cost_b:>11.4f}')

Cost per call (no rerank):   $0.000000
Cost per call (with rerank): $0.000219

Volume                  No Rerank  With Rerank
----------------------------------------------
100                  $     0.0000 $     0.0219
500                  $     0.0001 $     0.1093
1,000                $     0.0002 $     0.2186
5,000                $     0.0009 $     1.0929
10,000               $     0.0018 $     2.1857
50,000               $     0.0090 $    10.9285
100,000              $     0.0180 $    21.8570


In [14]:
# Ingest cost: what it costs to embed one new issue
# embed_text = TITLE + PROBLEM + RESOLUTION + COMMENTS (truncated to 8000 chars)
conn = get_conn()
cur = conn.cursor()
cur.execute("""
    SELECT title, body, resolution_text
    FROM issues
    WHERE body IS NOT NULL AND resolution_text IS NOT NULL
    ORDER BY RANDOM()
    LIMIT 50
""")
ingest_samples = cur.fetchall()
cur.close()
conn.close()

ingest_token_counts = []
for title, body, resolution in ingest_samples:
    embed_text = f"TITLE: {title}\nPROBLEM: {body}\nRESOLUTION: {resolution}"[:8000]
    ingest_token_counts.append(count_tokens(embed_enc, embed_text))

avg_ingest_tokens = sum(ingest_token_counts) / len(ingest_token_counts)
ingest_cost_per_issue = (avg_ingest_tokens / 1_000_000) * EMBEDDING_PRICE_PER_1M
current_issues = 46204

print(f'Avg tokens per issue embed:    {avg_ingest_tokens:.1f}')
print(f'Ingest cost per issue:         ${ingest_cost_per_issue:.6f}')
print(f'Cost to embed current {current_issues:,} issues: ${ingest_cost_per_issue * current_issues:.2f}')
print(f'Cost to embed 100k issues:     ${ingest_cost_per_issue * 100_000:.2f}')

Avg tokens per issue embed:    523.8
Ingest cost per issue:         $0.000010
Cost to embed current 46,204 issues: $0.48
Cost to embed 100k issues:     $1.05
